In [ ]:
#@title Install Dependencies
%%capture
!pip install gradio sentence-transformers chromadb openai -q

In [ ]:
#@title Set Your OpenAI API Key (Optional - enables AI-generated answers)
import os
from getpass import getpass

api_key = getpass("Enter your OpenAI API key (press Enter to skip): ")
if api_key.strip():
    os.environ["OPENAI_API_KEY"] = api_key
    print("API key set! RAG will use GPT-4o-mini with your chosen prompt patterns.")
else:
    print("No API key set. Using cached demo responses to show pattern differences.")

In [ ]:
#@title RAG Prompt Lab App
import gradio as gr
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import hashlib

# Initialize embedding model and ChromaDB
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))

# ── Sample Documents ─────────────────────────────────────────────────────────

SAMPLE_DOCS = {
    "Company HR Policy": """Remote Work Policy - Effective January 2024

Eligibility:
All full-time employees who have completed their 90-day probation period are eligible for remote work. Certain roles requiring physical presence (e.g., facilities, reception) are exempt.

Guidelines:
- Core hours: All remote employees must be available 10am-3pm in their local timezone for meetings and collaboration.
- Equipment: The company provides a laptop and monitor. Employees are responsible for reliable internet (minimum 25 Mbps).
- Workspace: Employees must have a dedicated workspace that allows for video calls without disruption.

Expectations:
- Respond to messages within 2 hours during core hours
- Attend all scheduled meetings with camera on
- Complete weekly status updates in the project management tool
- Be available for occasional in-office days (minimum 2 per month)

Expenses:
- Home office stipend: $500 one-time for setup
- Internet reimbursement: Up to $50/month
- Coworking space: Pre-approved expenses reimbursed

Violations of this policy may result in revocation of remote work privileges.""",

    "Product Returns FAQ": """Q: What is your return policy?
A: You can return most items within 30 days of purchase for a full refund. Items must be in original condition with tags attached. Electronics have a 15-day return window.

Q: How long does shipping take?
A: Standard shipping takes 5-7 business days. Express shipping takes 2-3 business days. Free shipping on orders over $50.

Q: How do I track my order?
A: Once your order ships, you'll receive an email with a tracking number. You can also log into your account to view order status.

Q: What payment methods do you accept?
A: We accept Visa, Mastercard, American Express, PayPal, and Apple Pay. All transactions are encrypted and secure.

Q: Can I exchange an item instead of returning it?
A: Yes, exchanges are available within 30 days. Visit any store location or contact support to initiate an exchange. Size exchanges on clothing are free.

Q: What if my item arrives damaged?
A: Contact us within 48 hours of delivery with photos of the damage. We will send a replacement at no cost and arrange return pickup of the damaged item.""",

    "AI Product Launch Guide": """AI Product Launch Guidelines - Internal

Phase 1 - Proof of Concept (PoC):
- Budget: Up to $25,000
- Duration: 4-6 weeks
- Success criteria: Accuracy above 80% on test set, latency under 2 seconds, positive feedback from 5+ internal users
- Red flags: If accuracy below 60% after 2 weeks, pivot or kill

Phase 2 - Pilot:
- Budget: $50,000-$100,000
- Duration: 8-12 weeks
- Success criteria: 90%+ user satisfaction, less than 5% error rate, clear ROI projection
- Red flags: User adoption below 30%, escalation rate above 20%

Phase 3 - Production:
- Budget: Varies by scale
- Requirements: Security review complete, monitoring dashboard live, rollback plan documented, on-call rotation established
- Launch checklist: Legal review, compliance sign-off, customer communication plan, support team trained

Cost Guidelines:
- LLM API costs: Budget $0.01-$0.10 per interaction
- Infrastructure: $500-$2000/month for vector DB + compute
- Always set cost alerts at 80% of monthly budget"""
}

SAMPLE_QUESTIONS = {
    "Company HR Policy": ["What are the core hours for remote employees?", "Can interns work remotely?", "What is the student discount?"],
    "Product Returns FAQ": ["What is the return window for electronics?", "Can I return opened software?", "Do you offer student discounts?"],
    "AI Product Launch Guide": ["What are the PoC success criteria?", "How long should a pilot take?", "What ML framework should we use?"]
}

# ── 6 Prompt Pattern Presets ──────────────────────────────────────────────────

PATTERNS = {
    "1. Bare (No Guardrails)": "Answer the question.\n\nContext:\n{context}\n\nQuestion: {question}",

    "2. Grounded Only": "Answer the question using ONLY the provided context. Do not use prior knowledge or make assumptions beyond what is explicitly stated.\n\nContext:\n{context}\n\nQuestion: {question}",

    "3. Grounded + Fallback": "Answer the question using ONLY the provided context. Do not use prior knowledge.\n\nIf the answer cannot be found in the provided context, say: \"I don't have information about that in the provided documents.\"\n\nContext:\n{context}\n\nQuestion: {question}",

    "4. Grounded + Fallback + Citations": "Answer the question using ONLY the provided context. Do not use prior knowledge.\n\nIf the answer cannot be found in the provided context, say: \"I don't have information about that in the provided documents.\"\n\nFor every factual claim, cite the source using [Chunk N] format.\n\nContext:\n{context}\n\nQuestion: {question}",

    "5. Grounded + Fallback + Citations + Fact Check": "Answer the question using ONLY the provided context. Do not use prior knowledge.\n\nIf the answer cannot be found in the provided context, say: \"I don't have information about that in the provided documents.\"\n\nFor every factual claim, cite the source using [Chunk N] format.\n\nAfter your answer, add a FACT CHECK section listing each claim you made and which chunk supports it. Flag any claims with weak support.\n\nContext:\n{context}\n\nQuestion: {question}",

    "6. Full Production (All Patterns)": "You are a knowledgeable assistant for this organization. Your role is to help employees and customers find accurate information.\n\nRULES:\n- Answer ONLY using the provided context. Do not use prior knowledge.\n- If the answer is not in the context, say: \"I don't have information about that in the provided documents. Let me connect you with a specialist.\"\n- Cite every factual claim using [Chunk N] format.\n- After your answer, include a FACT CHECK section listing each claim and its supporting chunk.\n- Review your response: Does every claim have a source? Remove any unsupported claims.\n- Never make promises about pricing, timelines, or policies not in the context.\n\nContext:\n{context}\n\nQuestion: {question}"
}

# ── Cached Demo Responses ─────────────────────────────────────────────────────

CACHED = {
    "Company HR Policy": {
        "What is the student discount?": {
            "1. Bare (No Guardrails)": "Based on the company policy, employees who are currently enrolled as students may be eligible for educational assistance benefits. The company typically offers a 15% discount on professional development courses and training materials. Students can also apply for flexible scheduling to accommodate their class schedules.",
            "3. Grounded + Fallback": "I don't have information about that in the provided documents. The policy covers remote work eligibility, guidelines, expectations, and expenses, but does not mention any student discounts.",
            "6. Full Production (All Patterns)": "I don't have information about that in the provided documents. Let me connect you with a specialist.\n\nThe provided context covers the Remote Work Policy including eligibility, guidelines, expectations, and expenses. No student discount program is mentioned in these documents.\n\n**FACT CHECK:**\n- Claim: No student discount is mentioned - Supported: verified by reviewing all chunks, none reference student discounts."
        },
        "What are the core hours for remote employees?": {
            "1. Bare (No Guardrails)": "The core hours for remote employees are typically 10am-3pm in their local timezone. This is a standard practice in many companies to ensure overlap for meetings and collaboration while still allowing flexibility in the morning and evening hours.",
            "4. Grounded + Fallback + Citations": "All remote employees must be available 10am-3pm in their local timezone for meetings and collaboration [Chunk 1].\n\nDuring these core hours, employees are expected to respond to messages within 2 hours [Chunk 2].",
            "6. Full Production (All Patterns)": "All remote employees must be available **10am-3pm in their local timezone** for meetings and collaboration [Chunk 1]. During core hours, the expectation is to respond to messages within 2 hours [Chunk 2].\n\n**FACT CHECK:**\n- Claim: Core hours are 10am-3pm local timezone - [Chunk 1] Supported\n- Claim: 2-hour response time during core hours - [Chunk 2] Supported"
        }
    },
    "Product Returns FAQ": {
        "Do you offer student discounts?": {
            "1. Bare (No Guardrails)": "Yes! We offer a 10% student discount on all regular-priced items. Simply verify your student status through our website using your .edu email address or a valid student ID. The discount is applied automatically at checkout once verified.",
            "3. Grounded + Fallback": "I don't have information about that in the provided documents. The FAQ covers return policies, shipping, order tracking, payment methods, exchanges, and damaged items, but does not mention student discounts.",
            "6. Full Production (All Patterns)": "I don't have information about that in the provided documents. Let me connect you with a specialist.\n\nThe available documentation covers: return policies, shipping timeframes, order tracking, payment methods, exchanges, and damaged item procedures. No student discount program is referenced.\n\n**FACT CHECK:**\n- Claim: No student discount mentioned in docs - Supported: reviewed all 6 FAQ entries, none reference student pricing."
        },
        "What is the return window for electronics?": {
            "1. Bare (No Guardrails)": "Electronics have a 15-day return window from the date of purchase. Make sure items are in their original condition with all accessories and packaging included for a full refund.",
            "4. Grounded + Fallback + Citations": "Electronics have a **15-day return window** [Chunk 1]. Items must be in original condition with tags attached for a full refund [Chunk 1].",
            "6. Full Production (All Patterns)": "Electronics have a **15-day return window** from the date of purchase [Chunk 1]. Items must be in original condition with tags attached to qualify for a full refund [Chunk 1].\n\nIf your electronic item arrives damaged, contact support within 48 hours with photos for a free replacement [Chunk 3].\n\n**FACT CHECK:**\n- Claim: 15-day return window for electronics - [Chunk 1] Supported\n- Claim: Original condition with tags required - [Chunk 1] Supported\n- Claim: 48-hour window for damaged items - [Chunk 3] Supported"
        }
    },
    "AI Product Launch Guide": {
        "What ML framework should we use?": {
            "1. Bare (No Guardrails)": "For most AI product launches, I'd recommend starting with PyTorch for research and prototyping, then considering TensorFlow for production deployment. If you're building NLP applications, Hugging Face Transformers is excellent. For classical ML, scikit-learn remains the gold standard.",
            "3. Grounded + Fallback": "I don't have information about that in the provided documents. The AI Product Launch Guidelines cover phases (PoC, Pilot, Production), budgets, success criteria, and cost guidelines, but do not recommend specific ML frameworks.",
            "6. Full Production (All Patterns)": "I don't have information about that in the provided documents. Let me connect you with a specialist.\n\nThe AI Product Launch Guidelines cover project phases, budgets, success criteria, red flags, and cost guidelines. Framework selection is not addressed in these documents - this would be a technical decision for your engineering team.\n\n**FACT CHECK:**\n- Claim: No ML framework recommendations in docs - Supported: reviewed all chunks covering PoC, Pilot, Production phases and cost guidelines. No framework mentioned."
        },
        "What are the PoC success criteria?": {
            "1. Bare (No Guardrails)": "The PoC success criteria typically include: accuracy above 80% on the test set, latency under 2 seconds, and positive feedback from internal users. You should also consider business metrics like potential ROI and user engagement.",
            "4. Grounded + Fallback + Citations": "The PoC success criteria are [Chunk 1]:\n- Accuracy above 80% on test set\n- Latency under 2 seconds\n- Positive feedback from 5+ internal users\n\nThe PoC budget is up to $25,000 with a 4-6 week duration [Chunk 1]. Red flag: if accuracy is below 60% after 2 weeks, pivot or kill the project [Chunk 1].",
            "6. Full Production (All Patterns)": "The PoC (Proof of Concept) success criteria are [Chunk 1]:\n\n1. **Accuracy** above 80% on test set\n2. **Latency** under 2 seconds\n3. **User feedback** - positive feedback from 5+ internal users\n\nAdditional PoC parameters:\n- Budget: Up to $25,000 [Chunk 1]\n- Duration: 4-6 weeks [Chunk 1]\n- Red flag: If accuracy below 60% after 2 weeks, pivot or kill [Chunk 1]\n\n**FACT CHECK:**\n- Claim: 80% accuracy threshold - [Chunk 1] Supported\n- Claim: 2-second latency target - [Chunk 1] Supported\n- Claim: 5+ internal user feedback - [Chunk 1] Supported\n- Claim: $25K budget cap - [Chunk 1] Supported\n- Claim: 4-6 week duration - [Chunk 1] Supported\n- Claim: 60% accuracy red flag at 2 weeks - [Chunk 1] Supported"
        }
    }
}

# ── Core RAG Backend ──────────────────────────────────────────────────────────

def chunk_text(text, chunk_size=400, overlap=80):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap
    return [c for c in chunks if c]

def process_documents(doc_text, chunk_size, overlap_pct):
    overlap = int(chunk_size * overlap_pct / 100)
    chunks = chunk_text(doc_text, chunk_size, overlap)
    collection_name = f"docs_{hashlib.md5(doc_text.encode()).hexdigest()[:8]}"
    try:
        chroma_client.delete_collection(collection_name)
    except:
        pass
    collection = chroma_client.create_collection(name=collection_name)
    embeddings = embed_model.encode(chunks).tolist()
    ids = [f"chunk_{i}" for i in range(len(chunks))]
    collection.add(embeddings=embeddings, documents=chunks, ids=ids)
    return collection, chunks

def retrieve_chunks(question, doc_text, chunk_size, overlap_pct, top_k):
    if not doc_text.strip():
        return [], [], []
    collection, chunks = process_documents(doc_text, chunk_size, overlap_pct)
    query_embedding = embed_model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=min(top_k, len(chunks)))
    return results['documents'][0], results['distances'][0], chunks

def format_context(retrieved_chunks):
    parts = []
    for i, chunk in enumerate(retrieved_chunks):
        parts.append(f"[Chunk {i+1}]:\n{chunk}")
    return "\n\n".join(parts)

def format_chunks_display(retrieved_chunks, distances):
    md = ""
    for i, (chunk, dist) in enumerate(zip(retrieved_chunks, distances)):
        similarity = max(0, 1 - dist)
        md += f"**Chunk {i+1}** (similarity: {similarity:.2f})\n```\n{chunk}\n```\n\n"
    return md

def generate_answer(system_prompt_template, question, context, temperature):
    prompt = system_prompt_template.replace("{context}", context).replace("{question}", question)
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        return None
    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=800
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"API Error: {str(e)}"

def get_cached_or_live(doc_name, question, pattern_name, system_prompt, context, temperature):
    # Try live API first
    live = generate_answer(system_prompt, question, context, temperature)
    if live:
        return live
    # Fall back to cache
    if doc_name in CACHED and question in CACHED[doc_name] and pattern_name in CACHED[doc_name][question]:
        return "**[Demo Mode - Cached Response]**\n\n" + CACHED[doc_name][question][pattern_name]
    return "**[Demo Mode]** No cached response for this combination. Enter an OpenAI API key for live responses."

# ── Tab 1: Single Query ───────────────────────────────────────────────────────

def run_single(doc_name, doc_text, question, pattern_name, chunk_size, overlap_pct, top_k, temperature):
    if not question.strip():
        return "Please enter a question.", "", ""
    retrieved, distances, _ = retrieve_chunks(question, doc_text, int(chunk_size), int(overlap_pct), int(top_k))
    if not retrieved:
        return "No documents loaded.", "", ""
    context = format_context(retrieved)
    system_prompt = PATTERNS[pattern_name]
    answer = get_cached_or_live(doc_name, question, pattern_name, system_prompt, context, temperature)
    chunks_display = format_chunks_display(retrieved, distances)
    stats = f"Pattern: {pattern_name} | Chunks: {len(retrieved)} | Top similarity: {max(0, 1-distances[0]):.2f} | Temperature: {temperature}"
    return answer, chunks_display, stats

# ── Tab 2: Compare Side-by-Side ───────────────────────────────────────────────

def run_compare(doc_name, doc_text, question, pattern_a, pattern_b, chunk_size, overlap_pct, top_k, temperature):
    if not question.strip():
        return "Enter a question.", "Enter a question.", "", ""
    retrieved, distances, _ = retrieve_chunks(question, doc_text, int(chunk_size), int(overlap_pct), int(top_k))
    if not retrieved:
        return "No documents.", "No documents.", "", ""
    context = format_context(retrieved)
    answer_a = get_cached_or_live(doc_name, question, pattern_a, PATTERNS[pattern_a], context, temperature)
    answer_b = get_cached_or_live(doc_name, question, pattern_b, PATTERNS[pattern_b], context, temperature)
    chunks_display = format_chunks_display(retrieved, distances)
    analysis = f"""## Key Differences\n\n| Aspect | Strategy A | Strategy B |\n|--------|-----------|-----------|\n| Pattern | {pattern_a} | {pattern_b} |\n| Same retrieval | Yes - identical chunks | Yes - identical chunks |\n\n> **The lesson:** Same documents, same retrieval, same model - only the **prompt pattern** changed. This is your cheapest lever for quality improvement."""
    return f"### Strategy A: {pattern_a}\n\n{answer_a}", f"### Strategy B: {pattern_b}\n\n{answer_b}", chunks_display, analysis

# ── Tab 3: Pattern Progression ────────────────────────────────────────────────

def run_progression(doc_name, doc_text, question, chunk_size, overlap_pct, top_k, temperature):
    if not question.strip():
        return "Enter a question.", ""
    retrieved, distances, _ = retrieve_chunks(question, doc_text, int(chunk_size), int(overlap_pct), int(top_k))
    if not retrieved:
        return "No documents.", ""
    context = format_context(retrieved)
    md = f"## All 6 Patterns for: *\"{question}\"*\n\n"
    md += format_chunks_display(retrieved, distances)
    md += "---\n\n"
    for name, template in PATTERNS.items():
        answer = get_cached_or_live(doc_name, question, name, template, context, temperature)
        md += f"### {name}\n\n{answer}\n\n---\n\n"
    summary = """## Summary\n\n| Pattern | Hallucination Risk | Citations | Handles Missing Info | Production Ready |\n|---------|-------------------|-----------|---------------------|-----------------|\n| 1. Bare | HIGH - no constraints | No | No - will fabricate | No |\n| 2. Grounded | Medium - constrained | No | Inconsistent | No |\n| 3. + Fallback | Low | No | Yes - declines | Minimum viable |\n| 4. + Citations | Low | Yes | Yes | Good |\n| 5. + Fact Check | Very Low | Yes + audit | Yes | Better |\n| 6. Full Production | Minimal | Yes + audit + review | Yes + escalation | Yes |\n\n> **PM Takeaway:** Each pattern adds ~1 line to the prompt but dramatically reduces risk. The jump from Bare to Grounded+Fallback is the highest ROI change you can make."""
    return md, summary

# ── Helper: load sample doc and questions ─────────────────────────────────────

def load_sample(doc_name):
    doc = SAMPLE_DOCS.get(doc_name, "")
    questions = SAMPLE_QUESTIONS.get(doc_name, ["", "", ""])
    return doc, gr.update(choices=questions, value=questions[0]), gr.update(choices=questions, value=questions[2]), gr.update(choices=questions, value=questions[2])

def load_pattern_prompt(pattern_name):
    return PATTERNS.get(pattern_name, "")

# ── Gradio UI ─────────────────────────────────────────────────────────────────

with gr.Blocks(title="RAG Prompt Lab", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# RAG Prompt Lab\n"
        "**Experiment with 6 prompt patterns and see how they transform RAG behavior.**\n\n"
        "> Same documents. Same retrieval. Same model. Only the **prompt pattern** changes - "
        "and that changes everything.\n\n"
        "API key may already be configured. If not, cached demos show the key differences."
    )

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Documents")
            doc_dropdown = gr.Dropdown(choices=list(SAMPLE_DOCS.keys()), value="Product Returns FAQ", label="Sample Document")
            doc_input = gr.Textbox(label="Document Text (paste your own or edit)", lines=8, value=SAMPLE_DOCS["Product Returns FAQ"])

            with gr.Accordion("RAG Configuration", open=False):
                chunk_size = gr.Slider(200, 1000, value=400, step=50, label="Chunk Size (chars)")
                overlap_pct = gr.Slider(0, 50, value=20, step=5, label="Overlap (%)")
                top_k = gr.Slider(1, 5, value=3, step=1, label="Chunks to Retrieve")
                temperature = gr.Slider(0.0, 1.0, value=0.0, step=0.1, label="Temperature")

        with gr.Column(scale=2):
            with gr.Tab("Single Query"):
                gr.Markdown("### Try one pattern at a time")
                t1_pattern = gr.Dropdown(choices=list(PATTERNS.keys()), value="1. Bare (No Guardrails)", label="Prompt Strategy")
                t1_prompt_preview = gr.Textbox(label="System Prompt (editable)", lines=4, value=PATTERNS["1. Bare (No Guardrails)"], interactive=True)
                t1_question = gr.Dropdown(choices=SAMPLE_QUESTIONS["Product Returns FAQ"], value=SAMPLE_QUESTIONS["Product Returns FAQ"][0], label="Question", allow_custom_value=True)
                t1_btn = gr.Button("Ask RAG", variant="primary")
                t1_answer = gr.Markdown(label="Answer")
                with gr.Accordion("Retrieved Chunks", open=False):
                    t1_chunks = gr.Markdown()
                t1_stats = gr.Textbox(label="Stats", interactive=False)

                t1_pattern.change(load_pattern_prompt, t1_pattern, t1_prompt_preview)
                t1_btn.click(run_single, [doc_dropdown, doc_input, t1_question, t1_pattern, chunk_size, overlap_pct, top_k, temperature], [t1_answer, t1_chunks, t1_stats])

            with gr.Tab("Compare Side-by-Side"):
                gr.Markdown("### A/B test two prompt strategies on the same question")
                t2_question = gr.Dropdown(choices=SAMPLE_QUESTIONS["Product Returns FAQ"], value=SAMPLE_QUESTIONS["Product Returns FAQ"][2], label="Question (try one NOT in the docs)", allow_custom_value=True)
                with gr.Row():
                    t2_pattern_a = gr.Dropdown(choices=list(PATTERNS.keys()), value="1. Bare (No Guardrails)", label="Strategy A")
                    t2_pattern_b = gr.Dropdown(choices=list(PATTERNS.keys()), value="3. Grounded + Fallback", label="Strategy B")
                t2_btn = gr.Button("Compare Side-by-Side", variant="primary")
                with gr.Row():
                    t2_answer_a = gr.Markdown()
                    t2_answer_b = gr.Markdown()
                with gr.Accordion("Retrieved Chunks (same for both)", open=False):
                    t2_chunks = gr.Markdown()
                t2_analysis = gr.Markdown()

                t2_btn.click(run_compare, [doc_dropdown, doc_input, t2_question, t2_pattern_a, t2_pattern_b, chunk_size, overlap_pct, top_k, temperature], [t2_answer_a, t2_answer_b, t2_chunks, t2_analysis])

            with gr.Tab("Pattern Progression"):
                gr.Markdown("### See all 6 patterns on the same question - watch quality improve")
                t3_question = gr.Dropdown(choices=SAMPLE_QUESTIONS["Product Returns FAQ"], value=SAMPLE_QUESTIONS["Product Returns FAQ"][2], label="Question (best with out-of-context questions)", allow_custom_value=True)
                t3_btn = gr.Button("Run All 6 Patterns", variant="primary")
                t3_results = gr.Markdown()
                t3_summary = gr.Markdown()

                t3_btn.click(run_progression, [doc_dropdown, doc_input, t3_question, chunk_size, overlap_pct, top_k, temperature], [t3_results, t3_summary])

    # Wire document dropdown to update doc text and question dropdowns
    doc_dropdown.change(load_sample, doc_dropdown, [doc_input, t1_question, t2_question, t3_question])

    gr.Markdown(
        "---\n"
        "**PM Takeaway:** Prompt engineering is your cheapest quality lever. "
        "Adding grounding + fallback + citations to a RAG system prompt costs nothing "
        "but dramatically reduces hallucination risk.\n\n"
        "*AI for Product Managers* | Patterns from Section 4.6"
    )

In [ ]:
#@title Launch App - Copy the gradio.live URL below
demo.launch(share=True)